In [1]:
import torch
from botorch.exceptions import InputDataWarning

from bott.problem import OptimizationProblem
from bott.physics_models import simulate_cbed
from bott.optimization import run_one_trial

import warnings
warnings.filterwarnings("ignore", category=InputDataWarning) 
# InputDataWarning: Data (outcome observations) is not standardized (std = tensor([5.3673e-11, 5.0607e-10, 4.8782e-10, 7.7036e-11, 5.1824e-14],
#        dtype=torch.float64), mean = tensor([0.0000e+00, 0.0000e+00, 8.4703e-22, 0.0000e+00, 0.0000e+00],
#        dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
#   check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)

In [2]:
ground_truth = torch.Tensor(simulate_cbed(20,10,-10, device_simu='cpu')) # abtem takes "cpu" or "gpu"

In [3]:
# OptimizationProblem would keep all the tensor on the specified device
problem = OptimizationProblem(ground_truth=ground_truth,
                              output_path='./output', 
                              save_results=True, 
                              reduction_params={'reduction_type':'square', 'reduction_kwargs':{'num_tiles':2}},
                              loss_params={'loss_type':'SSE', 'dp_pow': 0.5}, 
                              norm_arr=False,
                              dim=3, 
                              bounds=[(15,25), (-20, 20), (-20, 20)],
                              noise_std=0,
                              dtype=torch.float64, 
                              device='cpu'
                              ) # "cpu" or "cuda" for physics simulation

In [4]:
run_one_trial(problem_name='EICF', 
              problem=problem, 
              algo='EICF', 
              trial=5, 
              n_init_evals=2, 
              max_iter=50, 
              objective=None,
              dtype=torch.float64,
              device_botorch='cpu'
              )

Acquisition algo: EICF
Trial seed: 5
problem.device : cpu
GP model device: cpu
image output shape torch.Size([2, 315, 315])
Initializing model with '2' initial evaluations took 26.659 sec
y_temp tensor([[1.0149e-05, 1.0201e-05, 9.9825e-06, 1.0133e-05, 4.5418e-03]],
       dtype=torch.float64)

Iteration: 3/50
GP model fitting time: 0.316 sec
Acqu func sampling time: 1.621 sec
Physics simulation time: 14.431 sec
Iteration time: 16.372 sec
Suggested point: [[24.68316382 18.94333415  7.0737942 ]]
new Loss value: [[0.00454183]]
Reduction valued: tensor([[1.0149e-05, 1.0201e-05, 9.9825e-06, 1.0133e-05, 4.5418e-03]],
       dtype=torch.float64)
Best iteration index found: 2
Best point found: [24.11473373 14.74659886  1.56314731]
Best objective function value found: -0.0030060569234969282
y_temp tensor([[1.0036e-05, 1.0321e-05, 9.9069e-06, 1.0180e-05, 2.2831e-03]],
       dtype=torch.float64)

Iteration: 4/50
GP model fitting time: 0.254 sec
Acqu func sampling time: 2.415 sec
Physics simulati

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
image_test = torch.rand(2,315,315).to(torch.double)

In [ ]:
image_test= torch.cat((image_test,problem.measurement_true.unsqueeze(0)),dim=0)

In [ ]:
pixelLoss  = problem.loss_func(y_simu=image_test, y_true=problem.measurement_true, reduce=False).unsqueeze(-1) # pixelLoss = [n_init, 1]


In [ ]:
y_reduction = problem.reduction_func(image_test)

In [ ]:
y_reduction

In [ ]:
problem.reduction_true

In [ ]:
reductionLoss = problem.loss_func(y_simu=y_reduction, y_true=problem.reduction_true, reduce=False).unsqueeze(-1) # [n_init, 1]


In [ ]:
reductionLoss

In [ ]:
pixelLoss-reductionLoss

In [ ]:
pixelLoss